# 1. Data Processing and Visualizations


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import json
import os
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from wordcloud import WordCloud
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, roc_auc_score

## Text Cleaning Function


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
PRIORITY_TERMS = r'\b(dear|support|team|ni|hello|hope|well|customer|data|please|message|team|nan|could|would|assistance|problem|issue|failure|system|update)\b'
HTML_ARTIFACTS = r'\b(href|src|width|height|font|size|face|arial|sans|serif|color|border|style|nbsp|img|align|center|br|div|table|tr|td|span|strong|em)\b'

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(HTML_ARTIFACTS, '', text)
    text = re.sub(r'[\r\n\t\a\b]+', ' ', text)
    text = re.sub(PRIORITY_TERMS, '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)


## Load and Combine Datasets


In [ ]:
# Load Data
a = pd.read_csv('./raw_data_sets/aa_dataset-tickets-multi-lang-5-2-50-version.csv')
b = pd.read_csv('./raw_data_sets/dataset-tickets-multi-lang-4-20k.csv')
c = pd.read_csv('./raw_data_sets/email_dataset.csv')

a = a[a['language'] == 'en']
b = b[b['language'] == 'en']

base_cols = ['type', 'priority', 'body', 'subject']
tag_cols = [f'tag_{i}' for i in range(1, 9)]
req_cols = base_cols + tag_cols
a_sel = a[[col for col in req_cols if col in a.columns]]
b_sel = b[[col for col in req_cols if col in b.columns]]
tickets = pd.concat([a_sel, b_sel], ignore_index=True)

def process_type_row(row):
    curr = row.get('type', '')
    if curr in ['Incident', 'Change']:
        tags = [str(row.get(f'tag_{i}', '')).lower() for i in range(1, 9)]
        if 'feedback' in tags: return 'Feedback'
    if curr == 'Problem': return 'Complaint'
    return curr

tickets['type'] = tickets.apply(process_type_row, axis=1)
valid_types = ['Request', 'Complaint', 'Feedback']
tickets = tickets[tickets['type'].isin(valid_types)].copy()

c = c.rename(columns={'Label': 'type', 'Body': 'body', 'Subject': 'subject'})
c = c[c['type'] == 'spam'].copy()
c['priority'] = 'low'
for t in tag_cols: c[t] = np.nan

master = pd.concat([tickets[base_cols], c[base_cols]], ignore_index=True)
master.dropna(subset=['body', 'type', 'priority'], inplace=True)

min_cnt = master['type'].value_counts().min()
balanced = master.groupby('type', group_keys=False).apply(lambda x: x.sample(min_cnt, random_state=42)).reset_index(drop=True)
balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print('Data Loading Complete. Shape:', balanced.shape)
print(balanced['type'].value_counts())


## Visualizations


In [ ]:
fig = px.bar(balanced['type'].value_counts().reset_index(), x='type', y='count', title='Distribution of Email Types', color='type')
fig.show()

fig2 = px.bar(balanced['priority'].value_counts().reset_index(), x='priority', y='count', title='Distribution of Urgency', color='priority')
fig2.show()

balanced['body_length'] = balanced['body'].apply(lambda x: len(str(x).split()))
fig3 = px.histogram(balanced, x='body_length', color='type', title='Body Length Distribution by Type', nbins=50, barmode='overlay')
fig3.show()


In [ ]:
# Box Plot of Body Length by Type
fig4 = px.box(balanced, x='type', y='body_length', title='Distribution of Body Length by Email Type', color='type')
fig4.show()

# Time Series / Line Chart (Simulated if no timestamp exists)
# In standard ticket data without timestamps we simulate it over index to show volume trends
fig5 = px.line(balanced.reset_index(), x='index', y='body_length', title='Trend of Body Length across Dataset', color='type')
fig5.show()

In [ ]:

# Class Distribution Pie Chart
fig6 = px.pie(balanced, names='type', title='Proportion of Email Categories in Dataset', hole=0.3)
fig6.show()

# Scatter plot: Body Length vs Email Type, colored by Urgency
fig7 = px.scatter(balanced, x='type', y='body_length', color='priority', 
                  title='Email Type vs Body Length (Colored by Priority)', opacity=0.6,
                  category_orders={"priority": ["low", "medium", "high"]})
fig7.show()


## Clean and Save


In [ ]:
print('Cleaning Subject and Body...')
balanced['body_cleaned'] = balanced['body'].apply(clean_text)
balanced['subject_cleaned'] = balanced['subject'].apply(clean_text)

import os
os.makedirs('./cleaned_dataset_folder', exist_ok=True)
balanced.to_csv('./cleaned_dataset_folder/processed_dataset.csv', index=False)
print('Dataset saved.')


In [ ]:
# Word Clouds
def plot_wordcloud(text_series, title):
    text = ' '.join(text_series.dropna().astype(str))
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=16)
    plt.show()

print("Generating Word Cloud for Requests...")
plot_wordcloud(balanced[balanced['type'] == 'Request']['body_cleaned'], 'Top Words in Requests')

print("Generating Word Cloud for Complaints...")
plot_wordcloud(balanced[balanced['type'] == 'Complaint']['body_cleaned'], 'Top Words in Complaints')


# 2. Type Classification

Independent type training notebook.


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib
import os
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
PRIORITY_TERMS = r'\b(dear|support|team|ni|hello|hope|well|customer|data|please|message|team|nan|could|would|assistance|problem|issue|failure|system|update)\b'
HTML_ARTIFACTS = r'\b(href|src|width|height|font|size|face|arial|sans|serif|color|border|style|nbsp|img|align|center|br|div|table|tr|td|span|strong|em)\b'

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(HTML_ARTIFACTS, '', text)
    text = re.sub(r'[\r\n\t\a\b]+', ' ', text)
    text = re.sub(PRIORITY_TERMS, '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)


## Load Data


In [ ]:
if os.path.exists('./cleaned_dataset_folder/processed_dataset.csv'):
    print('Loading pre-cleaned dataset...')
    balanced = pd.read_csv('./cleaned_dataset_folder/processed_dataset.csv')
    balanced.dropna(subset=['body_cleaned', 'subject_cleaned', 'type', 'priority'], inplace=True)
else:
    print('Processed dataset not found. Loading and cleaning from raw data...')
    a = pd.read_csv('./raw_data_sets/aa_dataset-tickets-multi-lang-5-2-50-version.csv')
    b = pd.read_csv('./raw_data_sets/dataset-tickets-multi-lang-4-20k.csv')
    c = pd.read_csv('./raw_data_sets/email_dataset.csv')
    
    a = a[a['language'] == 'en']
    b = b[b['language'] == 'en']
    
    base_cols = ['type', 'priority', 'body', 'subject']
    tag_cols = [f'tag_{i}' for i in range(1, 9)]
    req_cols = base_cols + tag_cols
    a_sel = a[[col for col in req_cols if col in a.columns]]
    b_sel = b[[col for col in req_cols if col in b.columns]]
    tickets = pd.concat([a_sel, b_sel], ignore_index=True)
    
    def process_type_row(row):
        curr = row.get('type', '')
        if curr in ['Incident', 'Change']:
            tags = [str(row.get(f'tag_{i}', '')).lower() for i in range(1, 9)]
            if 'feedback' in tags: return 'Feedback'
        if curr == 'Problem': return 'Complaint'
        return curr
    
    tickets['type'] = tickets.apply(process_type_row, axis=1)
    valid_types = ['Request', 'Complaint', 'Feedback']
    tickets = tickets[tickets['type'].isin(valid_types)].copy()
    
    c = c.rename(columns={'Label': 'type', 'Body': 'body', 'Subject': 'subject'})
    c = c[c['type'] == 'spam'].copy()
    c['priority'] = 'low'
    for t in tag_cols: c[t] = np.nan
    
    master = pd.concat([tickets[base_cols], c[base_cols]], ignore_index=True)
    master.dropna(subset=['body', 'type', 'priority'], inplace=True)
    
    min_cnt = master['type'].value_counts().min()
    balanced = master.groupby('type', group_keys=False).apply(lambda x: x.sample(min_cnt, random_state=42)).reset_index(drop=True)
    balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print('Data Loading Complete. Shape:', balanced.shape)
    print(balanced['type'].value_counts())
    print('Cleaning Subjects and Bodies...')
    balanced['body_cleaned'] = balanced['body'].apply(clean_text)
    balanced['subject_cleaned'] = balanced['subject'].apply(clean_text)

print('Data Ready. Shape:', balanced.shape)


## Training Functions


In [ ]:
def train_and_eval(X, y, name):
    print(f'\n--- Training {name} ---')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
    X_train_vec = tfidf.fit_transform(X_train)
    X_test_vec = tfidf.transform(X_test)
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    print(f'{name} Accuracy: {accuracy_score(y_test, preds):.4f}')
    return model, tfidf, X_test, y_test


In [ ]:
print('=== TYPE CLASSIFICATION ENSEMBLE ===')
y_class = balanced['type']
X_body = balanced['body_cleaned'].fillna('')
X_subj = balanced['subject_cleaned'].fillna('')

c_body_model, c_body_tfidf, X_test_c_body, y_test_c = train_and_eval(X_body, y_class, 'Type (Body)')
c_subj_model, c_subj_tfidf, X_test_c_subj, _ = train_and_eval(X_subj, y_class, 'Type (Subject)')

probs_body = c_body_model.predict_proba(c_body_tfidf.transform(X_test_c_body))
probs_subj = c_subj_model.predict_proba(c_subj_tfidf.transform(X_test_c_subj))
avg_probs = (probs_body + probs_subj) / 2
final_preds_idx = np.argmax(avg_probs, axis=1)
final_preds = c_body_model.classes_[final_preds_idx]

print('\n--- Ensemble Type Results ---')
print(classification_report(y_test_c, final_preds))

os.makedirs('./models', exist_ok=True)
joblib.dump(c_body_model, './models/class_body_model.pkl')
joblib.dump(c_body_tfidf, './models/class_body_tfidf.pkl')
joblib.dump(c_subj_model, './models/class_subject_model.pkl')
joblib.dump(c_subj_tfidf, './models/class_subject_tfidf.pkl')
print('Type Models Saved!')


In [ ]:

# Confusion Matrix for Type Model Ensemble
cm_type = confusion_matrix(y_test_c, final_preds, labels=c_body_model.classes_)
disp_type = ConfusionMatrixDisplay(confusion_matrix=cm_type, display_labels=c_body_model.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp_type.plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix: Type Classification Ensemble')
plt.show()

# Evaluation Metrics for Type Model (Dummy encoded AUC)
try:
    y_test_c_dummy = pd.get_dummies(y_test_c)
    auc_score = roc_auc_score(y_test_c_dummy, avg_probs, multi_class='ovr')
    print(f"Overall ROC AUC Score (Type): {auc_score:.4f}")
except Exception as e:
    print("Could not compute AUC", e)


In [ ]:

# Top Predictive Keywords (Feature Importances) for Type Classification
def get_top_features(model, tfidf, n=10):
    feature_names = tfidf.get_feature_names_out()
    top_features = {}
    for i, class_label in enumerate(model.classes_):
        top_indices = np.argsort(model.coef_[i])[-n:]
        top_words = [feature_names[j] for j in top_indices]
        top_features[class_label] = top_words
    return pd.DataFrame(top_features)

print("\n--- Top 10 Keywords driving Type Predictions (Body) ---")
display(get_top_features(c_body_model, c_body_tfidf))


# 3. Urgency Classification

Independent urgency training notebook.


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib
import os
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
PRIORITY_TERMS = r'\b(dear|support|team|ni|hello|hope|well|customer|data|please|message|team|nan|could|would|assistance|problem|issue|failure|system|update)\b'
HTML_ARTIFACTS = r'\b(href|src|width|height|font|size|face|arial|sans|serif|color|border|style|nbsp|img|align|center|br|div|table|tr|td|span|strong|em)\b'

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(HTML_ARTIFACTS, '', text)
    text = re.sub(r'[\r\n\t\a\b]+', ' ', text)
    text = re.sub(PRIORITY_TERMS, '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)


## Load Data


In [ ]:
if os.path.exists('./cleaned_dataset_folder/processed_dataset.csv'):
    print('Loading pre-cleaned dataset...')
    balanced = pd.read_csv('./cleaned_dataset_folder/processed_dataset.csv')
    balanced.dropna(subset=['body_cleaned', 'subject_cleaned', 'type', 'priority'], inplace=True)
else:
    print('Processed dataset not found. Loading and cleaning from raw data...')
    a = pd.read_csv('./raw_data_sets/aa_dataset-tickets-multi-lang-5-2-50-version.csv')
    b = pd.read_csv('./raw_data_sets/dataset-tickets-multi-lang-4-20k.csv')
    c = pd.read_csv('./raw_data_sets/email_dataset.csv')
    
    a = a[a['language'] == 'en']
    b = b[b['language'] == 'en']
    
    base_cols = ['type', 'priority', 'body', 'subject']
    tag_cols = [f'tag_{i}' for i in range(1, 9)]
    req_cols = base_cols + tag_cols
    a_sel = a[[col for col in req_cols if col in a.columns]]
    b_sel = b[[col for col in req_cols if col in b.columns]]
    tickets = pd.concat([a_sel, b_sel], ignore_index=True)
    
    def process_type_row(row):
        curr = row.get('type', '')
        if curr in ['Incident', 'Change']:
            tags = [str(row.get(f'tag_{i}', '')).lower() for i in range(1, 9)]
            if 'feedback' in tags: return 'Feedback'
        if curr == 'Problem': return 'Complaint'
        return curr
    
    tickets['type'] = tickets.apply(process_type_row, axis=1)
    valid_types = ['Request', 'Complaint', 'Feedback']
    tickets = tickets[tickets['type'].isin(valid_types)].copy()
    
    c = c.rename(columns={'Label': 'type', 'Body': 'body', 'Subject': 'subject'})
    c = c[c['type'] == 'spam'].copy()
    c['priority'] = 'low'
    for t in tag_cols: c[t] = np.nan
    
    master = pd.concat([tickets[base_cols], c[base_cols]], ignore_index=True)
    master.dropna(subset=['body', 'type', 'priority'], inplace=True)
    
    min_cnt = master['type'].value_counts().min()
    balanced = master.groupby('type', group_keys=False).apply(lambda x: x.sample(min_cnt, random_state=42)).reset_index(drop=True)
    balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print('Data Loading Complete. Shape:', balanced.shape)
    print(balanced['type'].value_counts())
    print('Cleaning Subjects and Bodies...')
    balanced['body_cleaned'] = balanced['body'].apply(clean_text)
    balanced['subject_cleaned'] = balanced['subject'].apply(clean_text)

print('Data Ready. Shape:', balanced.shape)


## Training Functions


In [ ]:
def train_and_eval(X, y, name):
    print(f'\n--- Training {name} ---')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
    X_train_vec = tfidf.fit_transform(X_train)
    X_test_vec = tfidf.transform(X_test)
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    print(f'{name} Accuracy: {accuracy_score(y_test, preds):.4f}')
    return model, tfidf, X_test, y_test


In [ ]:
print('=== URGENCY ENSEMBLE ===')
y_urgency = balanced['priority']
X_body = balanced['body_cleaned'].fillna('')
X_subj = balanced['subject_cleaned'].fillna('')

u_body_model, u_body_tfidf, X_test_u_body, y_test_u = train_and_eval(X_body, y_urgency, 'Urgency (Body)')
u_subj_model, u_subj_tfidf, X_test_u_subj, _ = train_and_eval(X_subj, y_urgency, 'Urgency (Subject)')

probs_body = u_body_model.predict_proba(u_body_tfidf.transform(X_test_u_body))
probs_subj = u_subj_model.predict_proba(u_subj_tfidf.transform(X_test_u_subj))
avg_probs = (probs_body + probs_subj) / 2
final_preds_idx = np.argmax(avg_probs, axis=1)
final_preds = u_body_model.classes_[final_preds_idx]

print('\n--- Ensemble Urgency Results ---')
print(classification_report(y_test_u, final_preds))

os.makedirs('./models', exist_ok=True)
joblib.dump(u_body_model, './models/urgency_body_model.pkl')
joblib.dump(u_body_tfidf, './models/urgency_body_tfidf.pkl')
joblib.dump(u_subj_model, './models/urgency_subject_model.pkl')
joblib.dump(u_subj_tfidf, './models/urgency_subject_tfidf.pkl')
print('Urgency Models Saved!')


In [ ]:

# Confusion Matrix for Urgency Model Ensemble
cm_urg = confusion_matrix(y_test_u, final_preds, labels=u_body_model.classes_)
disp_urg = ConfusionMatrixDisplay(confusion_matrix=cm_urg, display_labels=u_body_model.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp_urg.plot(ax=ax, cmap='Oranges')
plt.title('Confusion Matrix: Urgency Classification Ensemble')
plt.show()

# Evaluation Metrics for Urgency Model (Dummy encoded AUC)
try:
    y_test_u_dummy = pd.get_dummies(y_test_u)
    auc_score = roc_auc_score(y_test_u_dummy, avg_probs, multi_class='ovr')
    print(f"Overall ROC AUC Score (Urgency): {auc_score:.4f}")
except Exception as e:
    print("Could not compute AUC", e)


In [ ]:

# Top Predictive Keywords (Feature Importances) for Urgency Classification
print("\n--- Top 10 Keywords driving Urgency Predictions (Body) ---")
display(get_top_features(u_body_model, u_body_tfidf))
